In [1]:
# CORRECTED FULL SCRIPT: enhanced_polynomial_model_with_ckm_cubic_up.py
"""
Enhanced Polynomial Mass Generation Model with CKM Matrix and CUBIC Up-Quark Fit

This module implements an enhanced version of the polynomial mass generation approach
that includes all six quarks, CKM matrix calculations, and uses a CUBIC polynomial
for the up-type quarks (u, c, t) to avoid a separate top quark enhancement term.

Author: Manus AI / Modified by Assistant
Date: April 22, 2025
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import pandas as pd # Keep pandas import as it might be used contextually later

class EnhancedPolynomialModelCubicUp:
    """
    An enhanced implementation of the polynomial mass generation approach
    that includes all six quarks, CKM matrix calculations, and uses a CUBIC
    polynomial fit for up-type quarks.
    """

    def __init__(self):
        """
        Initialize the enhanced polynomial model with lattice QCD parameters.
        """
        # Lattice QCD parameters (from provided papers and PDG)
        self.alpha_s_mz = 0.11803 # Strong coupling at Z-boson mass

        # Quark masses at reference scales (GeV)
        self.mc_mc = 1.2735 # Charm quark mass at its own scale
        self.mb_mb = 4.188 # Bottom quark mass at its own scale
        # PDG values for other quarks (GeV)
        self.mu_2gev = 0.00216 # Up quark mass at 2 GeV
        self.md_2gev = 0.00467 # Down quark mass at 2 GeV
        self.ms_2gev = 0.093 # Strange quark mass at 2 GeV
        self.mt_mt = 172.76 # Top quark mass at its own scale

        # Reference scales (GeV)
        self.mu_ref_light = 2.0 # Reference scale for light quarks (u, d, s)
        self.mu_c = self.mc_mc # Reference scale for charm quark
        self.mu_b = self.mb_mb # Reference scale for bottom quark
        self.mu_t = self.mt_mt # Reference scale for top quark
        self.mz = 91.1876 # Z-boson mass

        # --- Polynomial coefficients (to be optimized) ---
        # Up-type uses CUBIC (4 coeffs: a*L^3 + b*L^2 + c*L + d)
        self.poly_degree_up = 3
        self.c_coeffs_up_cubic = np.ones(self.poly_degree_up + 1) * 0.1 # Initial guess [a_u, b_u, c_u, d_u]

        # Down-type remains QUADRATIC (3 coeffs: a*L^2 + b*L + c)
        self.poly_degree_down = 2
        self.c_coeffs_down = np.ones(self.poly_degree_down + 1) * 0.1 # Initial guess [a_d, b_d, c_d]

        # Initialize geodesic lengths (to be optimized)
        self.L_u = 0.1 # Initial guess for up quark geodesic length
        self.L_d = 0.2 # Initial guess for down quark geodesic length
        self.L_s = 0.5 # Initial guess for strange quark geodesic length
        self.L_c = 1.0 # Initial guess for charm quark geodesic length
        self.L_b = 2.0 # Initial guess for bottom quark geodesic length
        self.L_t = 3.0 # Initial guess for top quark geodesic length

        # Initialize geodesic angles for CKM matrix (to be optimized)
        self.theta_12 = 0.2 # Initial guess for Cabibbo angle
        self.theta_13 = 0.01 # Initial guess for theta_13
        self.theta_23 = 0.04 # Initial guess for theta_23
        self.delta_cp = 1.2 # Initial guess for CP-violating phase

        # Experimental CKM matrix magnitudes (PDG 2022)
        self.ckm_exp = np.array([
            [0.97435, 0.22500, 0.00369],
            [0.22486, 0.97349, 0.04182],
            [0.00857, 0.04110, 0.99915]
        ])

        # Beta function coefficients (unchanged)
        self.beta0_nf3 = (11.0 - 2.0/3.0 * 3.0) / 4.0
        self.beta1_nf3 = (102.0 - 38.0/3.0 * 3.0) / 16.0
        self.beta0_nf4 = (11.0 - 2.0/3.0 * 4.0) / 4.0
        self.beta1_nf4 = (102.0 - 38.0/3.0 * 4.0) / 16.0
        self.beta0_nf5 = (11.0 - 2.0/3.0 * 5.0) / 4.0
        self.beta1_nf5 = (102.0 - 38.0/3.0 * 5.0) / 16.0
        self.beta0_nf6 = (11.0 - 2.0/3.0 * 6.0) / 4.0
        self.beta1_nf6 = (102.0 - 38.0/3.0 * 6.0) / 16.0

        # Anomalous dimension coefficients (unchanged)
        self.gamma0 = 1.0
        self.gamma1_nf3 = (202.0/3.0 - 20.0/9.0 * 3.0) / 16.0
        self.gamma1_nf4 = (202.0/3.0 - 20.0/9.0 * 4.0) / 16.0
        self.gamma1_nf5 = (202.0/3.0 - 20.0/9.0 * 5.0) / 16.0
        self.gamma1_nf6 = (202.0/3.0 - 20.0/9.0 * 6.0) / 16.0

        # Generation scaling factors (unchanged)
        self.gen_scale = [1.0, 1.0, 1.0] # To be optimized

        # Optimization status
        self.is_optimized = False
        self.results = {}

        # Quark information (unchanged)
        self.quark_info = {
            'u': {'type': 'up', 'generation': 1, 'ref_mass': self.mu_2gev, 'ref_scale': self.mu_ref_light},
            'd': {'type': 'down', 'generation': 1, 'ref_mass': self.md_2gev, 'ref_scale': self.mu_ref_light},
            's': {'type': 'down', 'generation': 2, 'ref_mass': self.ms_2gev, 'ref_scale': self.mu_ref_light},
            'c': {'type': 'up', 'generation': 2, 'ref_mass': self.mc_mc, 'ref_scale': self.mu_c},
            'b': {'type': 'down', 'generation': 3, 'ref_mass': self.mb_mb, 'ref_scale': self.mu_b},
            't': {'type': 'up', 'generation': 3, 'ref_mass': self.mt_mt, 'ref_scale': self.mu_t}
        }

    def alpha_s(self, mu):
        """Calculate the strong coupling constant at scale mu using 2-loop approximation."""
        # Determine number of active flavors
        if mu < 1.3: nf = 3; beta0 = self.beta0_nf3; beta1 = self.beta1_nf3
        elif mu < 4.2: nf = 4; beta0 = self.beta0_nf4; beta1 = self.beta1_nf4
        elif mu < 173.0: nf = 5; beta0 = self.beta0_nf5; beta1 = self.beta1_nf5
        else: nf = 6; beta0 = self.beta0_nf6; beta1 = self.beta1_nf6

        if abs(mu - self.mz) < 0.1: return self.alpha_s_mz

        if mu >= self.mz: # Run up from mz
             t = np.log(mu**2 / self.mz**2)
             # 2-loop running (simplified logic)
             as_mz_pi = self.alpha_s_mz / (4 * np.pi)
             term1 = 1 + beta0 * (4 * np.pi) * as_mz_pi * t
             # Add 2-loop correction (optional, check stability)
             # term2 = (beta1 / beta0) * np.log(term1) if term1 > 0 else 0
             # alpha_s_val = self.alpha_s_mz / (term1 + term2) if term1 > 0 else 0.1
             alpha_s_val = self.alpha_s_mz / term1 if term1 > 1e-6 else 0.1 # 1-loop stable version
             return max(0.01, alpha_s_val)

        elif mu >= 1.0: # Run down from mz (more complex, needs matching)
             # Simple approx: Use 1-loop running from mz down
             t = np.log(mu**2 / self.mz**2)
             if nf == 5: beta0_run = self.beta0_nf5
             elif nf == 4: beta0_run = self.beta0_nf4
             else: beta0_run = self.beta0_nf3

             alpha_s_inv = 1.0/self.alpha_s_mz + beta0_run * t
             return 1.0 / max(alpha_s_inv, 1e-6)

        else: # Freeze below 1 GeV
            return self.alpha_s(1.0)


    def running_mass(self, m_ref, mu_ref, mu, nf):
        """Calculate the running mass at scale mu."""
        # Select appropriate coefficients
        if nf == 3: gamma0 = self.gamma0; gamma1 = self.gamma1_nf3; beta0 = self.beta0_nf3
        elif nf == 4: gamma0 = self.gamma0; gamma1 = self.gamma1_nf4; beta0 = self.beta0_nf4
        elif nf == 5: gamma0 = self.gamma0; gamma1 = self.gamma1_nf5; beta0 = self.beta0_nf5
        else: gamma0 = self.gamma0; gamma1 = self.gamma1_nf6; beta0 = self.beta0_nf6

        if abs(mu - mu_ref) < 0.01: return m_ref

        # Ensure scales are positive and reasonable for alpha_s calculation
        mu_safe = max(mu, 0.1)
        mu_ref_safe = max(mu_ref, 0.1)

        try:
            alpha_ref = self.alpha_s(mu_ref_safe)
            alpha_mu = self.alpha_s(mu_safe)
            # Ensure alpha values are positive
            alpha_ref = max(alpha_ref, 1e-6)
            alpha_mu = max(alpha_mu, 1e-6)
        except (ValueError, OverflowError):
            return m_ref # Cannot calculate alpha_s, return reference mass

        # Use 1-loop running formula (most stable)
        try:
            if beta0 == 0: # Avoid division by zero
                power = 0
            else:
                power = gamma0 / (2 * beta0)

            ratio = alpha_mu / alpha_ref

            # Handle potential numerical issues with power calculation
            if ratio <= 0:
                 m_mu = m_ref # Avoid log of non-positive or invalid power base
            else:
                 m_mu = m_ref * (ratio**power)

            return max(m_mu, 1e-9) # Ensure positive mass

        except (ValueError, ZeroDivisionError, OverflowError):
            # Fallback: return reference mass if calculation fails
            return m_ref


    def calculate_ckm_matrix(self):
        """Calculate the CKM matrix using the standard parameterization."""
        s12, c12 = np.sin(self.theta_12), np.cos(self.theta_12)
        s13, c13 = np.sin(self.theta_13), np.cos(self.theta_13)
        s23, c23 = np.sin(self.theta_23), np.cos(self.theta_23)
        delta = self.delta_cp
        exp_neg_id = np.exp(-1j * delta)
        exp_id = np.exp(1j * delta)

        ckm = np.array([
            [c12 * c13, s12 * c13, s13 * exp_neg_id],
            [-s12 * c23 - c12 * s23 * s13 * exp_id, c12 * c23 - s12 * s23 * s13 * exp_id, s23 * c13],
            [s12 * s23 - c12 * c23 * s13 * exp_id, -c12 * s23 - s12 * c23 * s13 * exp_id, c23 * c13]
        ])
        return ckm

    def calculate_mass(self, quark, L=None):
        """
        Calculate quark mass from geodesic length using polynomial approach.
        Uses CUBIC for up-type, QUADRATIC for down-type.
        """
        quark_type = self.quark_info[quark]['type']
        generation = self.quark_info[quark]['generation']

        if L is None:
            # Safely get L value, defaulting to 0 if not set (e.g., before optimization)
            L = getattr(self, f'L_{quark}', 0.0)

        # Use parameters currently set in the instance
        current_gen_scale = getattr(self, 'gen_scale', [1.0, 1.0, 1.0])

        if quark_type == 'up':
            # Use cubic coefficients [a, b, c, d]
            coeffs = getattr(self, 'c_coeffs_up_cubic', [0.0] * (self.poly_degree_up + 1))
            # Calculate base mass using CUBIC polynomial: a*L^3 + b*L^2 + c*L + d
            mass = coeffs[0] * L**3 + coeffs[1] * L**2 + coeffs[2] * L + coeffs[3]

        else: # quark_type == 'down'
            # Use quadratic coefficients [a, b, c]
            coeffs = getattr(self, 'c_coeffs_down', [0.0] * (self.poly_degree_down + 1))
            # Calculate base mass using QUADRATIC polynomial: a*L^2 + b*L + c
            mass = coeffs[0] * L**2 + coeffs[1] * L + coeffs[2]

        # Apply generation-specific scaling
        # Check if generation index is valid for current_gen_scale
        if 0 <= generation - 1 < len(current_gen_scale):
             mass *= current_gen_scale[generation - 1]
        else:
             mass = 1e9 # Assign large mass if scaling factor is missing (error state)


        return max(mass, 1e-9) # Ensure mass is positive

    def optimize_parameters(self):
        """
        Optimize model parameters to match lattice QCD, PDG values, and CKM matrix.
        Uses CUBIC coefficients for up-type quarks.
        """
        def objective(params):
            # Unpack parameters safely
            try:
                # Number of parameters: 4(up) + 3(down) + 6(L) + 3(gen) + 4(CKM) = 20
                current_c_coeffs_up_cubic = params[0:4]    # Indices 0, 1, 2, 3
                current_c_coeffs_down = params[4:7]        # Indices 4, 5, 6
                current_L_u = params[7]                    # Index 7
                current_L_d = params[8]                    # Index 8
                current_L_s = params[9]                    # Index 9
                current_L_c = params[10]                   # Index 10
                current_L_b = params[11]                   # Index 11
                current_L_t = params[12]                   # Index 12
                current_gen_scale = params[13:16]          # Indices 13, 14, 15
                current_theta_12 = params[16]              # Index 16
                current_theta_13 = params[17]              # Index 17
                current_theta_23 = params[18]              # Index 18
                current_delta_cp = params[19]              # Index 19
            except IndexError:
                 return 1e20 # Return large error if parameter vector is wrong length

            # Temporarily update instance parameters for calculation within objective function
            # This is necessary because calculate_mass and calculate_ckm use instance attributes
            # Store original values to restore later (important!)
            original_params = {
                 'c_coeffs_up_cubic': self.c_coeffs_up_cubic, 'c_coeffs_down': self.c_coeffs_down,
                 'L_u': self.L_u, 'L_d': self.L_d, 'L_s': self.L_s, 'L_c': self.L_c, 'L_b': self.L_b, 'L_t': self.L_t,
                 'gen_scale': self.gen_scale, 'theta_12': self.theta_12, 'theta_13': self.theta_13,
                 'theta_23': self.theta_23, 'delta_cp': self.delta_cp
            }

            self.c_coeffs_up_cubic = current_c_coeffs_up_cubic
            self.c_coeffs_down = current_c_coeffs_down
            self.L_u, self.L_d, self.L_s = current_L_u, current_L_d, current_L_s
            self.L_c, self.L_b, self.L_t = current_L_c, current_L_b, current_L_t
            self.gen_scale = current_gen_scale
            self.theta_12, self.theta_13 = current_theta_12, current_theta_13
            self.theta_23, self.delta_cp = current_theta_23, current_delta_cp

            # Calculate masses and errors
            masses = {}
            errors = {}
            total_error = 0
            for quark in self.quark_info:
                ref_mass = self.quark_info[quark]['ref_mass']
                try:
                    # calculate_mass now uses the temporarily updated instance parameters
                    masses[quark] = self.calculate_mass(quark)
                    if ref_mass > 1e-9:
                         error_term = ((masses[quark] - ref_mass) / ref_mass)**2
                    else:
                         error_term = (masses[quark] / 1e-3)**2 # Penalize deviation from zero (scaled)

                    # Apply weights (same as original)
                    weight = 1.0
                    if quark in ['c', 'b']: weight = 5.0
                    elif quark == 't': weight = 20.0

                    errors[quark] = error_term # Store unweighted squared fractional error
                    total_error += weight * error_term

                except (ValueError, OverflowError):
                    total_error += 1e10 # Large penalty if calculation fails
                    errors[quark] = 1e10
                    masses[quark] = np.inf


            # Calculate CKM matrix and errors
            try:
                ckm = self.calculate_ckm_matrix() # Uses updated angles
                ckm_mag = np.abs(ckm)
                if np.any(np.isnan(ckm_mag)) or np.any(np.isinf(ckm_mag)):
                    ckm_term_error = 1e10 # Large penalty for invalid CKM
                else:
                    # Avoid division by zero in CKM errors for small elements
                    ckm_exp_safe = np.maximum(self.ckm_exp, 1e-6)
                    ckm_errors = ((ckm_mag - self.ckm_exp) / ckm_exp_safe)**2
                    ckm_term_error = 10.0 * np.sum(ckm_errors) # Weight CKM error sum
            except (ValueError, OverflowError, np.linalg.LinAlgError):
                 ckm_term_error = 1e10

            total_error += ckm_term_error


            # Regularization to prevent extreme parameter values
            try:
                reg_poly_up = np.sum(current_c_coeffs_up_cubic**2)
                reg_poly_down = np.sum(current_c_coeffs_down**2)
                current_Ls = np.array([current_L_u, current_L_d, current_L_s, current_L_c, current_L_b, current_L_t])
                initial_Ls = np.array([0.1, 0.2, 0.5, 1.0, 2.0, 3.0])
                reg_L = np.sum((current_Ls - initial_Ls)**2)
                reg_gen = np.sum((np.array(current_gen_scale) - 1.0)**2)
                reg_ckm = (current_theta_12 - 0.2)**2 + (current_theta_13 - 0.01)**2 + \
                          (current_theta_23 - 0.04)**2 + (current_delta_cp - 1.2)**2

                regularization = 0.01 * (reg_poly_up + reg_poly_down + reg_L + reg_gen + reg_ckm)
            except (ValueError, OverflowError):
                 regularization = 1e10 # Penalize if regularization calc fails


            # Restore original instance parameters
            self.c_coeffs_up_cubic = original_params['c_coeffs_up_cubic']
            self.c_coeffs_down = original_params['c_coeffs_down']
            self.L_u, self.L_d, self.L_s = original_params['L_u'], original_params['L_d'], original_params['L_s']
            self.L_c, self.L_b, self.L_t = original_params['L_c'], original_params['L_b'], original_params['L_t']
            self.gen_scale = original_params['gen_scale']
            self.theta_12, self.theta_13 = original_params['theta_12'], original_params['theta_13']
            self.theta_23, self.delta_cp = original_params['theta_23'], original_params['delta_cp']

            final_objective = total_error + regularization
            # Ensure total error is a finite number
            if np.isnan(final_objective) or np.isinf(final_objective):
                return 1e20 # Return a large finite number

            return final_objective

        # Initial guess vector (20 parameters)
        initial_guess = np.concatenate([
            self.c_coeffs_up_cubic,     # 4 parameters
            self.c_coeffs_down,         # 3 parameters
            [self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t], # 6 parameters
            self.gen_scale,             # 3 parameters
            [self.theta_12, self.theta_13, self.theta_23, self.delta_cp] # 4 parameters
        ])

        # Bounds list (should have 20 tuples)
        bounds = []
        # Bounds for up-type CUBIC coefficients [a_u, b_u, c_u, d_u]
        bounds.extend([(-100.0, 100.0)] * (self.poly_degree_up + 1))
        # Bounds for down-type QUADRATIC coefficients [a_d, b_d, c_d]
        bounds.extend([(-100.0, 100.0)] * (self.poly_degree_down + 1))

        # Geodesic lengths
        bounds.append((0.01, 0.5)) # L_u
        bounds.append((0.01, 0.5)) # L_d
        bounds.append((0.1, 1.5)) # L_s (wider)
        bounds.append((0.5, 3.0)) # L_c (wider)
        bounds.append((1.0, 6.0)) # L_b (wider)
        bounds.append((2.0, 10.0)) # L_t (wider)

        # Generation scaling factors
        bounds.append((0.001, 20.0))  # gen_scale[0] (wider)
        bounds.append((0.01, 100.0))  # gen_scale[1] (wider)
        bounds.append((0.1, 1000.0)) # gen_scale[2] (wider)

        # CKM parameters (same as original)
        bounds.append((0.1, 0.3)) # theta_12
        bounds.append((0.001, 0.05)) # theta_13
        bounds.append((0.01, 0.1)) # theta_23
        bounds.append((0.0, 2*np.pi)) # delta_cp

        # Check bounds length
        if len(bounds) != len(initial_guess):
             raise ValueError(f"Mismatch between number of parameters ({len(initial_guess)}) and bounds ({len(bounds)})")


        # Perform optimization with increased limits
        print(f"Starting optimization with L-BFGS-B...")
        result = minimize(objective, initial_guess, method='L-BFGS-B', bounds=bounds,
                          options={'maxiter': 50000, 'maxfun': 50000, 'ftol': 1e-9, 'gtol': 1e-6})

        # Update instance parameters with optimized values ONLY IF successful
        if result.success:
            print("Optimization successful.")
            optimized_params = result.x
            self.c_coeffs_up_cubic = optimized_params[0:4]
            self.c_coeffs_down = optimized_params[4:7]
            self.L_u = optimized_params[7]
            self.L_d = optimized_params[8]
            self.L_s = optimized_params[9]
            self.L_c = optimized_params[10]
            self.L_b = optimized_params[11]
            self.L_t = optimized_params[12]
            self.gen_scale = optimized_params[13:16]
            self.theta_12 = optimized_params[16]
            self.theta_13 = optimized_params[17]
            self.theta_23 = optimized_params[18]
            self.delta_cp = optimized_params[19]
            self.is_optimized = True
            final_objective_value = result.fun
        else:
            print(f"Optimization failed: {result.message}")
            # Keep initial parameters if optimization failed
            self.is_optimized = False
            # Calculate objective with initial guess for reporting
            final_objective_value = objective(initial_guess)


        # Calculate final masses and errors with optimized (or initial) params
        final_masses = {}
        final_errors_percent = {}
        for quark in self.quark_info:
            ref_mass = self.quark_info[quark]['ref_mass']
            pred_mass = self.calculate_mass(quark) # Uses current instance params
            final_masses[quark] = pred_mass
            if ref_mass > 1e-9:
                 final_errors_percent[quark] = abs((pred_mass - ref_mass) / ref_mass) * 100 # Percentage
            else:
                 final_errors_percent[quark] = abs(pred_mass) * 100 # Error relative to zero

        # Calculate final CKM matrix and errors
        final_ckm = self.calculate_ckm_matrix() # Uses current instance angles
        final_ckm_mag = np.abs(final_ckm)
        ckm_exp_safe = np.maximum(self.ckm_exp, 1e-6)
        final_ckm_errors_percent = np.abs((final_ckm_mag - self.ckm_exp) / ckm_exp_safe) * 100 # Percentage

        # Store results
        self.results = {
            'c_coeffs_up_cubic': self.c_coeffs_up_cubic,
            'c_coeffs_down': self.c_coeffs_down,
            'L_u': self.L_u, 'L_d': self.L_d, 'L_s': self.L_s,
            'L_c': self.L_c, 'L_b': self.L_b, 'L_t': self.L_t,
            'gen_scale': self.gen_scale,
            'theta_12': self.theta_12, 'theta_13': self.theta_13,
            'theta_23': self.theta_23, 'delta_cp': self.delta_cp,
            'masses': final_masses,
            'errors_percent': final_errors_percent, # Store percentage errors
            'ckm': final_ckm_mag,
            'ckm_exp': self.ckm_exp,
            'ckm_errors_percent': final_ckm_errors_percent, # Store percentage errors
            'success': self.is_optimized, # Use the final status flag
            'message': result.message if hasattr(result, 'message') else "Optimization not run or failed early",
            'final_objective_value': final_objective_value
        }

        return self.results


    def calculate_running_masses(self, mu_values):
        """Calculate running masses at different energy scales."""
        if not self.is_optimized:
            print("Warning: Parameters not optimized. Cannot calculate running masses accurately.")
            # Return NaN values if not optimized
            return {'mu': mu_values, **{q: [np.nan]*len(mu_values) for q in self.quark_info}}

        running_masses = {'mu': mu_values}
        for quark in self.quark_info:
            running_masses[quark] = []
            # Use the OPTIMIZED mass at its reference scale as m_ref
            ref_mass_pred = self.results['masses'][quark]
            ref_scale = self.quark_info[quark]['ref_scale']
            for mu in mu_values:
                # Determine number of active flavors (consistent logic)
                if mu < 1.3: nf = 3
                elif mu < 4.2: nf = 4
                elif mu < 173.0: nf = 5
                else: nf = 6

                # Calculate running mass based on the predicted reference mass
                m_mu = self.running_mass(ref_mass_pred, ref_scale, mu, nf)
                running_masses[quark].append(m_mu)
        return running_masses

    def calculate_alpha_s_values(self, mu_values):
        """Calculate strong coupling constant at different energy scales."""
        alpha_s_values = {'mu': mu_values, 'alpha_s': []}
        for mu in mu_values:
            alpha_s_values['alpha_s'].append(self.alpha_s(mu))
        return alpha_s_values

    def generate_report(self):
        """Generate a report of the model results (updated for cubic and corrected CKM output)."""
        # Ensure results exist, even if optimization failed (uses initial params then)
        if not self.results:
            print("Warning: No results found. Running optimization with initial parameters.")
            self.optimize_parameters() # Run optimization to populate results

        report_path = "enhanced_model_cubic_up_report.md"
        print(f"Generating report: {report_path}")
        with open(report_path, 'w') as f:
            f.write("# Enhanced Polynomial Model with CKM Matrix (Cubic Up-Quark Fit) Report\n\n")
            f.write("## Optimization Status\n\n")
            f.write(f"Success: {self.results.get('success', False)}\n") # Default to False
            f.write(f"Message: {self.results.get('message', 'N/A')}\n")
            f.write(f"Final Objective Function Value: {self.results.get('final_objective_value', 'N/A'):.6e}\n\n")

            f.write("## Optimized Parameters (or Initial if Failed)\n\n")
            f.write("### Polynomial Coefficients for Up-type Quarks (Cubic: aL^3 + bL^2 + cL + d)\n\n")
            labels = ['a_u', 'b_u', 'c_u', 'd_u']
            coeffs_up = self.results.get('c_coeffs_up_cubic', [np.nan]*4)
            for i, c in enumerate(coeffs_up):
                f.write(f"{labels[i]} = {c:.6f} \n")

            f.write("\n### Polynomial Coefficients for Down-type Quarks (Quadratic: aL^2 + bL + c)\n\n")
            labels_down = ['a_d', 'b_d', 'c_d']
            coeffs_down = self.results.get('c_coeffs_down', [np.nan]*3)
            for i, c in enumerate(coeffs_down):
                f.write(f"{labels_down[i]} = {c:.6f} \n")

            f.write("\n### Generation Scaling Factors\n\n")
            gen_scale = self.results.get('gen_scale', [np.nan]*3)
            for i, s in enumerate(gen_scale):
                f.write(f"Generation {i+1}: {s:.6f}\n")

            f.write("\n### Geodesic Lengths\n\n")
            for q in ['u', 'd', 's', 'c', 'b', 't']:
                 l_val = self.results.get(f'L_{q}', np.nan)
                 f.write(f"L_{q} = {l_val:.6f}\n")

            f.write("\n### CKM Matrix Parameters\n\n")
            theta_12 = self.results.get('theta_12', np.nan)
            theta_13 = self.results.get('theta_13', np.nan)
            theta_23 = self.results.get('theta_23', np.nan)
            delta_cp = self.results.get('delta_cp', np.nan)
            # Add checks for NaN before calculating degrees
            deg12 = np.degrees(theta_12) if not np.isnan(theta_12) else np.nan
            deg13 = np.degrees(theta_13) if not np.isnan(theta_13) else np.nan
            deg23 = np.degrees(theta_23) if not np.isnan(theta_23) else np.nan
            degCP = np.degrees(delta_cp) if not np.isnan(delta_cp) else np.nan
            f.write(f"θ₁₂ = {theta_12:.6f} rad = {deg12:.4f}°\n")
            f.write(f"θ₁₃ = {theta_13:.6f} rad = {deg13:.4f}°\n")
            f.write(f"θ₂₃ = {theta_23:.6f} rad = {deg23:.4f}°\n")
            f.write(f"δ_CP = {delta_cp:.6f} rad = {degCP:.4f}°\n")

            f.write("\n## Mass Predictions at Reference Scales\n\n")
            f.write("| Quark | Reference Scale (GeV) | Predicted Mass (GeV) | Reference Value (GeV) | Error (%) |\n")
            f.write("|-------|----------------------|----------------------|----------------------|----------|\n")
            masses = self.results.get('masses', {})
            errors_percent = self.results.get('errors_percent', {})
            for quark in self.quark_info:
                ref_scale = self.quark_info[quark]['ref_scale']
                ref_mass = self.quark_info[quark]['ref_mass']
                pred_mass = masses.get(quark, np.nan)
                error = errors_percent.get(quark, np.nan)
                f.write(f"| {quark:<5} | {ref_scale:20.4f} | {pred_mass:20.6f} | {ref_mass:20.6f} | {error:8.4f} |\n")


            f.write("\n## CKM Matrix\n\n")

            # --- CORRECTED CKM OUTPUT ---
            f.write("### Predicted CKM Matrix (Magnitudes)\n\n")
            f.write("```
            ckm_pred = self.results.get('ckm', np.full((3,3), np.nan))
            for i in range(3):
                line = "[ " + " ".join([f"{ckm_pred[i, j]:.6f}" for j in range(3)]) + " ]\n"
                f.write(line) # Write each formatted line
            f.write("```\n\n") # End code block

            f.write("### Experimental CKM Matrix (Magnitudes)\n\n")
            f.write("```
            for i in range(3):
                line = "[ " + " ".join([f"{self.ckm_exp[i, j]:.6f}" for j in range(3)]) + " ]\n"
                f.write(line)
            f.write("```\n\n") # End code block

            f.write("### CKM Matrix Errors (%)\n\n")
            f.write("```
            ckm_err_percent = self.results.get('ckm_errors_percent', np.full((3,3), np.nan))
            for i in range(3):
                line = "[ " + " ".join([f"{ckm_err_percent[i, j]:.4f}" for j in range(3)]) + " ]\n"
                f.write(line)
            f.write("```\n\n") # End code block
            # --- END CORRECTED CKM OUTPUT ---


            f.write("\n## Running Masses\n\n")
            # Check if optimization was successful before calculating/writing this
            if self.results.get('success', False):
                 mu_values_report = [1.0, 2.0, 5.0, 10.0, self.mz, self.mt_mt]
                 running_masses = self.calculate_running_masses(mu_values_report)
                 f.write("| μ (GeV) | m_u (GeV) | m_d (GeV) | m_s (GeV) | m_c (GeV) | m_b (GeV) | m_t (GeV) |\n")
                 f.write("|---------|-----------|-----------|-----------|-----------|-----------|----------|\n")
                 for i, mu in enumerate(running_masses['mu']):
                      # Check for NaN before formatting
                      m_u = running_masses['u'][i]; m_d = running_masses['d'][i]; m_s = running_masses['s'][i]
                      m_c = running_masses['c'][i]; m_b = running_masses['b'][i]; m_t = running_masses['t'][i]
                      f.write(f"| {mu:7.4f} | "
                              f"{m_u:9.6f if not np.isnan(m_u) else ' NaN':<9} | "
                              f"{m_d:9.6f if not np.isnan(m_d) else ' NaN':<9} | "
                              f"{m_s:9.6f if not np.isnan(m_s) else ' NaN':<9} | "
                              f"{m_c:9.6f if not np.isnan(m_c) else ' NaN':<9} | "
                              f"{m_b:9.6f if not np.isnan(m_b) else ' NaN':<9} | "
                              f"{m_t:9.6f if not np.isnan(m_t) else ' NaN':<9} |\n")
            else:
                 f.write("Running masses not calculated (Optimization failed or not run).\n")


            f.write("\n## Strong Coupling Constant\n\n")
            if self.results.get('success', False):
                 mu_values_report = [1.0, 2.0, 5.0, 10.0, self.mz, self.mt_mt] # Use same values
                 alpha_s_values = self.calculate_alpha_s_values(mu_values_report)
                 f.write("| μ (GeV) | α_s |\n")
                 f.write("|---------|------|\n")
                 for i, mu in enumerate(alpha_s_values['mu']):
                      alpha = alpha_s_values['alpha_s'][i]
                      f.write(f"| {mu:7.4f} | {alpha:.6f if not np.isnan(alpha) else ' NaN'} |\n")
            else:
                 f.write("Alpha_s values not calculated (Optimization failed or not run).\n")

        return report_path

    # --- Plotting functions ---
    # These functions generally use the data stored in self.results after optimization.
    # Ensure they handle potential NaNs or missing data gracefully if optimization fails.

    def plot_running_masses(self):
        """Plot running masses (requires successful optimization)."""
        if not self.is_optimized:
            print("Warning: Cannot plot running masses. Optimization failed or not run.")
            return None

        fig, ax = plt.subplots(figsize=(10, 6))
        mu_values_plot = np.logspace(0, 3, 100) # 1 GeV to 1000 GeV
        running_masses = self.calculate_running_masses(mu_values_plot)

        if running_masses is None: return None # Exit if calculation failed

        for quark in self.quark_info:
             if quark in running_masses and not np.all(np.isnan(running_masses[quark])):
                 ax.loglog(running_masses['mu'], running_masses[quark], label=f'{quark}')

        # Add reference points (Predicted masses at ref scale)
        for quark in self.quark_info:
             ref_scale = self.quark_info[quark]['ref_scale']
             pred_mass = self.results.get('masses', {}).get(quark, np.nan)
             if not np.isnan(pred_mass) and not np.isnan(ref_scale):
                  ax.scatter([ref_scale], [pred_mass], marker='o', s=30, label=f'_{quark} pred')

        ax.set_title('Running Quark Masses (Predicted)')
        ax.set_xlabel('Energy Scale μ (GeV)'); ax.set_ylabel('Running Mass (GeV)')
        ax.grid(True, which='both', linestyle='--', alpha=0.7)
        ax.legend(fontsize='small')
        ax.set_ylim(bottom=1e-3) # Set reasonable lower limit for log plot

        plot_path = "enhanced_cubic_running_masses_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        return plot_path


    def plot_polynomial_functions(self):
        """Plot the polynomial mass generation functions (updated for cubic)."""
        # Plot even if optimization failed, using current parameters (likely initial)
        plot_path_base = "enhanced_poly_cubic_functions"
        if not self.results: self.optimize_parameters() # Ensure results exist

        coeffs_up = self.results.get('c_coeffs_up_cubic', [0,0,0,0])
        coeffs_down = self.results.get('c_coeffs_down', [0,0,0])
        gen_scales = self.results.get('gen_scale', [1,1,1])
        Ls = {q: self.results.get(f'L_{q}', np.nan) for q in self.quark_info}
        pred_masses = self.results.get('masses', {})

        # Determine range for L based on optimized L_t or a default
        L_max = Ls.get('t', 5.0)
        if np.isnan(L_max): L_max = 5.0
        L_values = np.linspace(0, L_max * 1.1, 200)

        # Calculate polynomial values
        up_base_masses = coeffs_up[0] * L_values**3 + coeffs_up[1] * L_values**2 + coeffs_up[2] * L_values + coeffs_up[3]
        down_base_masses = coeffs_down[0] * L_values**2 + coeffs_down[1] * L_values + coeffs_down[2]

        # --- Linear Scale Plot ---
        fig_lin, ax_lin = plt.subplots(figsize=(10, 6))

        # Plot scaled polynomial curves
        colors = ['r', 'b']
        styles = ['--', '-.', '-']
        labels_poly = [['Up Poly*g1', 'Up Poly*g2', 'Up Poly*g3 (Cubic)'],
                       ['Down Poly*g1', 'Down Poly*g2', 'Down Poly*g3 (Quad)']]

        for i in range(3): # Generations
             # Up type
             mass_vals_up = np.maximum(up_base_masses * gen_scales[i], -1e9) # Allow negative for linear plot viz
             ax_lin.plot(L_values, mass_vals_up, color=colors[0], linestyle=styles[i], alpha=0.6, label=labels_poly[0][i])
             # Down type
             mass_vals_down = np.maximum(down_base_masses * gen_scales[i], -1e9)
             ax_lin.plot(L_values, mass_vals_down, color=colors[1], linestyle=styles[i], alpha=0.6, label=labels_poly[1][i])

        # Plot individual quark points
        markers = ['o', 's', '^'] # Gen 1, 2, 3
        for quark, info in self.quark_info.items():
            q_L = Ls.get(quark, np.nan)
            q_mass = pred_masses.get(quark, np.nan)
            q_gen_idx = info['generation'] - 1
            q_type_idx = 0 if info['type'] == 'up' else 1
            if not np.isnan(q_L) and not np.isnan(q_mass):
                 ax_lin.scatter([q_L], [q_mass], color=colors[q_type_idx], marker=markers[q_gen_idx], s=60, label=f'{quark} (pred)', zorder=5)

        ax_lin.set_xlabel('Geodesic Length L')
        ax_lin.set_ylabel('Predicted Mass (GeV)')
        ax_lin.set_title('Polynomial Mass Generation Functions (Scaled)')
        ax_lin.grid(True, linestyle='--', alpha=0.7)
        ax_lin.legend(fontsize='small', ncol=2)
        max_abs_mass = max(abs(m) for m in pred_masses.values() if m is not None and not np.isnan(m))
        if max_abs_mass > 0: ax_lin.set_ylim(bottom=-max_abs_mass*0.1)

        plot_path_lin = f"{plot_path_base}_plot.png"
        plt.savefig(plot_path_lin, dpi=300, bbox_inches='tight')
        plt.close(fig_lin)

        # --- Log Scale Plot ---
        fig_log, ax_log = plt.subplots(figsize=(10, 6))

        # Plot scaled polynomial curves (ensure positive for log)
        for i in range(3): # Generations
             mass_vals_up_log = np.maximum(up_base_masses * gen_scales[i], 1e-9)
             ax_log.semilogy(L_values, mass_vals_up_log, color=colors[0], linestyle=styles[i], alpha=0.6, label=labels_poly[0][i])
             mass_vals_down_log = np.maximum(down_base_masses * gen_scales[i], 1e-9)
             ax_log.semilogy(L_values, mass_vals_down_log, color=colors[1], linestyle=styles[i], alpha=0.6, label=labels_poly[1][i])

        # Plot individual quark points
        for quark, info in self.quark_info.items():
            q_L = Ls.get(quark, np.nan)
            q_mass = pred_masses.get(quark, np.nan)
            q_gen_idx = info['generation'] - 1
            q_type_idx = 0 if info['type'] == 'up' else 1
            if not np.isnan(q_L) and not np.isnan(q_mass) and q_mass > 1e-9: # Check positive for log
                 ax_log.scatter([q_L], [q_mass], color=colors[q_type_idx], marker=markers[q_gen_idx], s=60, label=f'{quark} (pred)', zorder=5)

        ax_log.set_xlabel('Geodesic Length L')
        ax_log.set_ylabel('Predicted Mass (GeV) - Log Scale')
        ax_log.set_title('Polynomial Mass Generation Functions (Scaled, Log Scale)')
        ax_log.grid(True, which='both', linestyle='--', alpha=0.7)
        ax_log.legend(fontsize='small', ncol=2)

        # Adjust y-lim for log scale if possible
        valid_masses = [m for m in pred_masses.values() if m is not None and not np.isnan(m) and m > 1e-9]
        if valid_masses:
             min_mass_log = min(valid_masses)
             max_mass_log = max(valid_masses)
             ax_log.set_ylim(bottom=min_mass_log * 0.1, top=max_mass_log * 10)
        else:
             ax_log.set_ylim(bottom=1e-3, top=1e3) # Default log limits

        plot_path_log = f"{plot_path_base}_log_plot.png"
        plt.savefig(plot_path_log, dpi=300, bbox_inches='tight')
        plt.close(fig_log)

        return plot_path_lin, plot_path_log

    def plot_geodesic_lengths(self):
        """Plot the geodesic lengths for all quarks."""
        if not self.results: self.optimize_parameters()

        fig, ax = plt.subplots(figsize=(10, 6))
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        lengths = [self.results.get(f'L_{q}', np.nan) for q in quarks]
        colors = ['red', 'blue', 'blue', 'red', 'blue', 'red'] # Color by type

        valid_indices = [i for i, l in enumerate(lengths) if not np.isnan(l)]
        quarks_valid = [quarks[i] for i in valid_indices]
        lengths_valid = [lengths[i] for i in valid_indices]
        colors_valid = [colors[i] for i in valid_indices]

        if lengths_valid:
             ax.bar(quarks_valid, lengths_valid, color=colors_valid)
             ax.set_xlabel('Quark')
             ax.set_ylabel('Optimized Geodesic Length (L)')
             ax.set_title('Geodesic Lengths')
             ax.grid(True, axis='y', linestyle='--', alpha=0.7)
        else:
             ax.set_title("Geodesic Lengths - No valid data")

        plot_path = "enhanced_cubic_geodesic_lengths_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        return plot_path

    def plot_mass_hierarchy(self):
        """Plot the mass hierarchy for all quarks."""
        if not self.results: self.optimize_parameters()

        fig, ax = plt.subplots(figsize=(10, 6))
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        masses = [self.results.get('masses', {}).get(q, np.nan) for q in quarks]
        colors = ['red', 'blue', 'blue', 'red', 'blue', 'red'] # Color by type

        valid_indices = [i for i, m in enumerate(masses) if not np.isnan(m) and m > 1e-9] # Positive for log
        quarks_valid = [quarks[i] for i in valid_indices]
        masses_valid = [masses[i] for i in valid_indices]
        colors_valid = [colors[i] for i in valid_indices]

        if masses_valid:
             ax.bar(quarks_valid, masses_valid, color=colors_valid)
             ax.set_yscale('log')
             ax.set_xlabel('Quark')
             ax.set_ylabel('Predicted Mass (GeV) - Log Scale')
             ax.set_title('Quark Mass Hierarchy (Predicted)')
             ax.grid(True, axis='y', which='both', linestyle='--', alpha=0.7)
        else:
             ax.set_title("Quark Mass Hierarchy (Predicted) - No valid data")

        plot_path = "enhanced_cubic_mass_hierarchy_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        return plot_path

    def plot_ckm_matrix(self):
        """Plot the CKM matrix comparison."""
        if not self.results: self.optimize_parameters()

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        ckm_pred = self.results.get('ckm', np.full((3,3), np.nan))
        ckm_err_percent = self.results.get('ckm_errors_percent', np.full((3,3), np.nan))

        # Find max error for color scale, handle all NaNs
        max_err = 20.0 # Default max
        if not np.all(np.isnan(ckm_err_percent)):
             max_err = max(max_err, np.nanmax(ckm_err_percent))

        # Plot 1: Predicted CKM matrix
        im1 = axes[0].imshow(ckm_pred, cmap='viridis', vmin=0, vmax=1)
        axes[0].set_title('Predicted CKM Matrix (Magnitudes)')
        for i in range(3):
            for j in range(3):
                 val = ckm_pred[i, j]
                 axes[0].text(j, i, f"{val:.4f}" if not np.isnan(val) else "NaN",
                              ha="center", va="center", color="w" if val < 0.5 else "k")

        # Plot 2: Experimental CKM matrix
        im2 = axes[1].imshow(self.ckm_exp, cmap='viridis', vmin=0, vmax=1)
        axes[1].set_title('Experimental CKM Matrix (Magnitudes)')
        for i in range(3):
            for j in range(3):
                 val = self.ckm_exp[i, j]
                 axes[1].text(j, i, f"{val:.4f}", ha="center", va="center", color="w" if val < 0.5 else "k")

        # Plot 3: Error percentage
        im3 = axes[2].imshow(ckm_err_percent, cmap='hot', vmin=0, vmax=max_err)
        axes[2].set_title('Error Percentage (%)')
        for i in range(3):
            for j in range(3):
                 err = ckm_err_percent[i, j]
                 axes[2].text(j, i, f"{err:.2f}%" if not np.isnan(err) else "NaN",
                              ha="center", va="center", color="w" if err > max_err*0.5 else "k")

        for ax in axes:
            ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
            ax.set_xticklabels(['d', 's', 'b']); ax.set_yticklabels(['u', 'c', 't'])
            ax.set_xlabel('Down-type Quarks'); ax.set_ylabel('Up-type Quarks')

        fig.colorbar(im1, ax=axes[0], label='Magnitude', shrink=0.7);
        fig.colorbar(im2, ax=axes[1], label='Magnitude', shrink=0.7);
        fig.colorbar(im3, ax=axes[2], label='Error (%)', shrink=0.7)
        plt.tight_layout()
        plot_path = "enhanced_cubic_ckm_matrix_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        return plot_path


    def create_comprehensive_visualization(self):
        """Create a comprehensive visualization (updated for cubic)."""
        if not self.results: self.optimize_parameters()

        fig = plt.figure(figsize=(15, 12))
        # Add title only if optimized successfully
        fig_title = "Comprehensive Model Results (Cubic Up-Quark Fit)"
        if not self.is_optimized: fig_title += " - OPTIMIZATION FAILED"
        fig.suptitle(fig_title, fontsize=16)

        # --- Plot 1: Running masses ---
        ax1 = fig.add_subplot(2, 2, 1)
        if self.is_optimized:
             mu_values_plot = np.logspace(0, 3, 100)
             running_masses = self.calculate_running_masses(mu_values_plot)
             if running_masses is not None:
                 for quark in self.quark_info:
                      if quark in running_masses and not np.all(np.isnan(running_masses[quark])):
                          ax1.loglog(running_masses['mu'], running_masses[quark], label=f'{quark}')
                 # Add reference points
                 for quark in self.quark_info:
                      ref_scale = self.quark_info[quark]['ref_scale']
                      pred_mass = self.results.get('masses', {}).get(quark, np.nan)
                      if not np.isnan(pred_mass) and not np.isnan(ref_scale):
                          ax1.scatter([ref_scale], [pred_mass], marker='o', s=30, label=f'_{quark} pred')
                 ax1.set_ylim(bottom=1e-3)
                 ax1.legend(fontsize='small')
             else:
                 ax1.text(0.5, 0.5, "Running masses calculation failed", ha='center', va='center')
             ax1.set_title('Running Quark Masses (Predicted)')
             ax1.set_xlabel('Energy Scale μ (GeV)'); ax1.set_ylabel('Running Mass (GeV)')
             ax1.grid(True, which='both', linestyle='--', alpha=0.7)
        else:
            ax1.text(0.5, 0.5, "Running masses not plotted\n(Optimization failed)", ha='center', va='center')
            ax1.set_title('Running Quark Masses (Predicted)')


        # --- Plot 2: CKM matrix ---
        ax2 = fig.add_subplot(2, 2, 2)
        ckm_pred = self.results.get('ckm', np.full((3,3), np.nan))
        im = ax2.imshow(ckm_pred, cmap='viridis', vmin=0, vmax=1)
        ax2.set_title('Predicted CKM Matrix (Magnitudes)')
        for i in range(3):
            for j in range(3):
                 val = ckm_pred[i, j]
                 ax2.text(j, i, f"{val:.4f}" if not np.isnan(val) else "NaN",
                          ha="center", va="center", color="w" if val < 0.5 else "k")
        ax2.set_xticks([0, 1, 2]); ax2.set_yticks([0, 1, 2])
        ax2.set_xticklabels(['d', 's', 'b']); ax2.set_yticklabels(['u', 'c', 't'])
        ax2.set_xlabel('Down-type'); ax2.set_ylabel('Up-type')
        fig.colorbar(im, ax=ax2, label='Magnitude', shrink=0.8)


        # --- Plot 3: Polynomial functions (Log Scale) ---
        ax3 = fig.add_subplot(2, 2, 3)
        coeffs_up = self.results.get('c_coeffs_up_cubic', [0,0,0,0])
        coeffs_down = self.results.get('c_coeffs_down', [0,0,0])
        gen_scales = self.results.get('gen_scale', [1,1,1])
        Ls = {q: self.results.get(f'L_{q}', np.nan) for q in self.quark_info}
        pred_masses = self.results.get('masses', {})
        L_max = Ls.get('t', 5.0); L_values = np.linspace(0, L_max * 1.1, 100)
        up_base = coeffs_up[0]*L_values**3 + coeffs_up[1]*L_values**2 + coeffs_up[2]*L_values + coeffs_up[3]
        down_base = coeffs_down[0]*L_values**2 + coeffs_down[1]*L_values + coeffs_down[2]

        colors = ['r', 'b']; styles = ['--', '-.', '-']
        for i in range(3): # Generations
             ax3.semilogy(L_values, np.maximum(up_base * gen_scales[i], 1e-9), color=colors[0], linestyle=styles[i], alpha=0.6, label=f'Up Poly*g{i+1}')
             ax3.semilogy(L_values, np.maximum(down_base * gen_scales[i], 1e-9), color=colors[1], linestyle=styles[i], alpha=0.6, label=f'Down Poly*g{i+1}')

        markers = ['o', 's', '^']
        for quark, info in self.quark_info.items():
            q_L = Ls.get(quark, np.nan); q_mass = pred_masses.get(quark, np.nan)
            if not np.isnan(q_L) and not np.isnan(q_mass) and q_mass > 1e-9:
                 ax3.scatter([q_L], [q_mass], color=colors[0 if info['type']=='up' else 1],
                             marker=markers[info['generation']-1], s=60, label=f'{quark} (pred)', zorder=5)

        ax3.set_xlabel('Geodesic Length L')
        ax3.set_ylabel('Predicted Mass (GeV) - Log Scale')
        ax3.set_title('Polynomial Functions (Scaled)')
        ax3.grid(True, which='both', linestyle='--', alpha=0.7)
        ax3.legend(fontsize='small', ncol=2)
        valid_masses = [m for m in pred_masses.values() if m is not None and not np.isnan(m) and m > 1e-9]
        if valid_masses: ax3.set_ylim(min(valid_masses)*0.1, max(valid_masses)*10)
        else: ax3.set_ylim(1e-3, 1e3)


        # --- Plot 4: Mass hierarchy ---
        ax4 = fig.add_subplot(2, 2, 4)
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        masses = [self.results.get('masses', {}).get(q, np.nan) for q in quarks]
        colors = ['red', 'blue', 'blue', 'red', 'blue', 'red']

        valid_indices = [i for i, m in enumerate(masses) if not np.isnan(m) and m > 1e-9]
        quarks_valid = [quarks[i] for i in valid_indices]
        masses_valid = [masses[i] for i in valid_indices]
        colors_valid = [colors[i] for i in valid_indices]

        if masses_valid:
            ax4.bar(quarks_valid, masses_valid, color=colors_valid)
            ax4.set_yscale('log')
            ax4.set_title('Quark Mass Hierarchy (Predicted)')
            ax4.set_xlabel('Quark'); ax4.set_ylabel('Mass (GeV) - Log Scale')
            ax4.grid(True, axis='y', which='both', linestyle='--', alpha=0.7)
        else:
            ax4.text(0.5, 0.5, "Mass hierarchy not plotted\n(No valid mass data)", ha='center', va='center')
            ax4.set_title('Quark Mass Hierarchy (Predicted)')

        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout
        plot_path = "enhanced_cubic_comprehensive_visualization.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        return plot_path


# --- Main execution block ---
if __name__ == "__main__":
    # Create model instance
    model = EnhancedPolynomialModelCubicUp()

    # Optimize parameters
    # print("Starting optimization...") # Moved inside optimize_parameters
    results = model.optimize_parameters()
    # print(f"Optimization finished. Success: {model.is_optimized}") # Status printed inside optimize_parameters
    # print(f"Message: {results.get('message')}")
    # print(f"Final Objective Value: {results.get('final_objective_value', 'N/A')}")

    # Generate report and plots
    # These functions now check internally if optimization succeeded or use initial params
    print("\nGenerating report and plots (using optimized parameters if successful)...")

    try:
         report_path = model.generate_report()
         if report_path: print(f"Report generated: {report_path}")
    except Exception as e:
        print(f"Error generating report: {e}")

    # Plotting functions
    plot_functions = [
        model.plot_running_masses,
        model.plot_polynomial_functions,
        model.plot_geodesic_lengths,
        model.plot_mass_hierarchy,
        model.plot_ckm_matrix,
        model.create_comprehensive_visualization
    ]

    for plot_func in plot_functions:
        try:
            plot_paths = plot_func()
            if plot_paths:
                 # Handle functions returning single or multiple paths
                 if isinstance(plot_paths, tuple):
                     for p in plot_paths: print(f"Plot generated: {p}")
                 else:
                      print(f"Plot generated: {plot_paths}")
        except Exception as e:
            print(f"Error generating plot with {plot_func.__name__}: {e}")


    # --- Print summary (optional, report contains this info) ---
    print("\n--- Summary from Results ---")
    print(f"Optimization Success: {model.is_optimized}")
    print(f"Optimization Message: {results.get('message')}")

    print("\nMass Predictions (% Error):")
    errors_percent = results.get('errors_percent', {})
    for quark in model.quark_info:
        error = errors_percent.get(quark, np.nan)
        print(f"  {quark}: {error:.4f}%")

    print("\nCKM Errors (%):")
    ckm_err_percent = results.get('ckm_errors_percent', np.full((3,3), np.nan))
    for i in range(3):
        print(f"  [{' '.join([f'{x:.4f}' if not np.isnan(x) else ' NaN ' for x in ckm_err_percent[i,:]])}]")

    print("\nFinished.")


SyntaxError: unterminated string literal (detected at line 580) (<ipython-input-1-b064461f8d28>, line 580)